In [1]:
import pandas as pd



In [49]:
orders=pd.read_csv(r"C:\KSR AIML Course\Data Files\Pandas\PandasPractice_09092026\orders.csv")
customers=pd.read_csv(r"C:\KSR AIML Course\Data Files\Pandas\PandasPractice_09092026\customers.csv")
products=pd.read_csv(r"C:\KSR AIML Course\Data Files\Pandas\PandasPractice_09092026\products.csv")

In [4]:
orders.head()

,order_id,line_no,order_date,customer_id,product_id,quantity,unit_price,discount_pct,payment_method,order_status,delivery_days,rating,sales_channel
0,O005545,1,2026-02-18,C01881,P0133,2,3911.00,0.10,Debit Card,Delivered,5.0,NaN,Web
1,O005690,1,2026-04-16,C00939,P0054,1,13221.43,0.15,Debit Card,Delivered,3.0,4.0,Web
2,O005472,3,2026-07-29,C00360,P0006,2,21007.01,0.25,credit card,Delivered,5.0,3.0,web
3,O004310,2,2025-04-06,C01670,P0015,1,21646.09,0.00,upi,Delivered,6.0,3.0,APP
4,O006198,1,2025-03-01,C00515,P0143,2,4154.68,0.20,credit card,Delivered,7.0,3.0,Marketplace


In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 11535 entries, 0 to 11534
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        11535 non-null  str    
 1   line_no         11535 non-null  int64  
 2   order_date      11535 non-null  str    
 3   customer_id     11535 non-null  str    
 4   product_id      11535 non-null  str    
 5   quantity        11535 non-null  int64  
 6   unit_price      11535 non-null  float64
 7   discount_pct    11535 non-null  float64
 8   payment_method  11401 non-null  str    
 9   order_status    11535 non-null  str    
 10  delivery_days   10582 non-null  float64
 11  rating          9949 non-null   float64
 12  sales_channel   11535 non-null  str    
dtypes: float64(4), int64(2), str(7)
memory usage: 1.1 MB


In [7]:
orders.nunique()

order_id           6500
line_no               4
order_date          608
customer_id        1931
product_id          144
quantity              5
unit_price        11389
discount_pct          9
payment_method        8
order_status          3
delivery_days        11
rating                5
sales_channel         5
dtype: int64

In [26]:
def quality_report(df):
    return pd.DataFrame({
        "dtype":df.dtypes.astype("str"),
        "missing_count":df.isnull().sum(),
        "missing_percentage":(df.isnull().mean()*100).round(2),
        "unique_values":df.nunique(dropna=False)
    }).sort_values("missing_percentage",ascending=False)
display(quality_report(orders))
print("Exact Duplicate records count:",orders.duplicated().sum())
print("Payment methods",orders["payment_method"].value_counts(dropna=False))
print("Sales_channel",orders["sales_channel"].value_counts(dropna=False))

,dtype,missing_count,missing_percentage,unique_values
rating,float64,1586,13.75,6
delivery_days,float64,953,8.26,12
payment_method,str,134,1.16,9
line_no,int64,0,0.00,4
order_id,str,0,0.00,6500
product_id,str,0,0.00,144
customer_id,str,0,0.00,1931
order_date,str,0,0.00,608
quantity,int64,0,0.00,5
discount_pct,float64,0,0.00,9


Exact Duplicate records count: 75
Payment methods payment_method
upi                 1485
Debit Card          1444
Credit Card         1442
credit card         1439
Wallet              1428
Cash on Delivery    1393
UPI                 1385
COD                 1385
NaN                  134
Name: count, dtype: int64
Sales_channel sales_channel
Web            2341
web            2339
App            2309
Marketplace    2281
APP            2265
Name: count, dtype: int64


In [74]:
## 2. Cleaning
orders["order_date"]=pd.to_datetime(orders["order_date"])
customers["signup_date"]=pd.to_datetime(customers["signup_date"])
products["launch_date"]=pd.to_datetime(products["launch_date"])

clean_orders=orders.drop_duplicates().copy()

payment_map={
    "UPI":"UPI","upi":"UPI",
    "Credit Card":"Credit Card","credit card":"Credit Card",
    "Wallet":"Wallet",
    "COD":"Cash on Delivery","Cash on Delivery":"Cash on Delivery",
    "Debit Card":"Debit Card"

}
clean_orders["payment_method"]=clean_orders["payment_method"].map(payment_map).fillna("unknown")

sales_channel_map={
    "Web":"Web","web":"Web",
    "APP":"App","App":"App",
    "Marketplace":"Marketplace"
}
clean_orders["sales_channel"]=clean_orders["sales_channel"].map(sales_channel_map).fillna("unknown")

clean_orders["delivery_days"]=pd.to_numeric(clean_orders["delivery_days"],errors="coerce").astype("Int64")
clean_orders["rating"]=pd.to_numeric(clean_orders["rating"],errors="coerce").astype("Int64")

customers["age"]=pd.to_numeric(customers["age"],errors="coerce").astype("Int64")
customers["age"]=customers['age'].fillna(customers["age"].median())
customers["city"]=customers["city"].fillna("unknown")

clean_orders["zero_price_falg"]=clean_orders["unit_price"].le(0)
clean_orders["high_discount_flag"]=clean_orders["discount_pct"].ge(0.50)

clean_orders=clean_orders.loc[~clean_orders["zero_price_falg"]].copy()

print("Rows before clean data",len(orders))
print("Rows for after clean data analysis",len(clean_orders))

Rows before clean data 11535
Rows for after clean data analysis 11419


In [69]:
customers.sample(10)
customers.query("customer_id=='C01196'")

,customer_id,customer_name,signup_date,age,gender,city,state,segment,acquisition_channel
1195,C01196,Sai Nair,2025-07-28,65,Male,unknown,Andhra Pradesh,Consumer,Direct


In [73]:
clean_orders

,order_id,line_no,order_date,customer_id,product_id,quantity,unit_price,discount_pct,payment_method,order_status,delivery_days,rating,sales_channel,zero_price_falg,high_discount_flag
0,O005545,1,2026-02-18,C01881,P0133,2,3911.00,0.10,Debit Card,Delivered,5,<NA>,Web,False,False
1,O005690,1,2026-04-16,C00939,P0054,1,13221.43,0.15,Debit Card,Delivered,3,4,Web,False,False
2,O005472,3,2026-07-29,C00360,P0006,2,21007.01,0.25,Credit Card,Delivered,5,3,Web,False,False
3,O004310,2,2025-04-06,C01670,P0015,1,21646.09,0.00,UPI,Delivered,6,3,App,False,False
4,O006198,1,2025-03-01,C00515,P0143,2,4154.68,0.20,Credit Card,Delivered,7,3,Marketplace,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11530,O005441,1,2025-02-07,C00473,P0022,2,5537.37,0.25,Cash on Delivery,Returned,4,4,Web,False,False
11531,O001567,1,2025-06-16,C00669,P0091,3,258.26,0.20,UPI,Delivered,4,4,Web,False,False
11532,O003837,2,2025-03-17,C00780,P0019,2,25549.77,0.20,UPI,Delivered,6,3,Web,False,False
11533,O004502,1,2025-10-14,C00979,P0084,3,16305.81,0.05,UPI,Returned,1,5,Web,False,False
